# TRIBE v2 — Backend on Colab
**Run every cell top to bottom. Cell 1 will auto-restart the runtime — that is expected.**

In [ ]:
# ── CELL 1: Fix numpy + auto-restart ──────────────────────────────────────
# Must run first and alone. Runtime will restart automatically.
!pip install -q numpy==2.2.6 --force-reinstall
print('numpy fixed. Restarting runtime now...')
import os, time; time.sleep(2)
os.kill(os.getpid(), 9)

In [ ]:
# ── CELL 2: Install all deps (after restart) ──────────────────────────────
!apt-get install -y -q ffmpeg

# torch 2.8.0 already on Colab — do not reinstall unless missing
import subprocess, sys
r = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                   capture_output=True, text=True)
if '2.8' not in r.stdout:
    print('Upgrading torch...')
    !pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
else:
    print(f'torch OK: {r.stdout.strip()}')

!pip install -q whisperx fastapi 'uvicorn[standard]' yt-dlp pydantic nest_asyncio huggingface_hub

import numpy as np, torch
print(f'numpy: {np.__version__}  (need 2.2.6)')
print(f'torch: {torch.__version__}')
assert np.__version__ == '2.2.6', f'numpy wrong: {np.__version__} — re-run Cell 1'

In [ ]:
# ── CELL 3: Clone + install tribev2 ───────────────────────────────────────
import os
if not os.path.exists('/content/tribev2'):
    !git clone -q https://github.com/facebookresearch/tribev2 /content/tribev2
else:
    !git -C /content/tribev2 pull -q
    print('tribev2 already present, pulled latest')

# Install without letting pip clobber numpy
!pip install -q -e /content/tribev2 --no-deps
# Reinstall any tribev2 deps that aren't numpy/torch (already correct)
!pip install -q torchaudio einops omegaconf

# Verify tribev2 imports
import tribev2
print(f'tribev2 OK — {tribev2.__file__}')

In [ ]:
# ── CELL 4: Discover tribev2 API ──────────────────────────────────────────
import tribev2, os, inspect, pkgutil, importlib

pkg_dir = os.path.dirname(tribev2.__file__)

print('=== Python files ===')
for root, dirs, files in os.walk(pkg_dir):
    dirs[:] = [d for d in dirs if d != '__pycache__']
    for f in files:
        if f.endswith('.py'):
            print(' ', os.path.relpath(os.path.join(root, f), pkg_dir))

print('\n=== Classes + methods ===')
for importer, modname, ispkg in pkgutil.walk_packages(tribev2.__path__, prefix='tribev2.'):
    try:
        mod = importlib.import_module(modname)
        for cname, cls in inspect.getmembers(mod, inspect.isclass):
            if cls.__module__ and cls.__module__.startswith('tribev2'):
                methods = [n for n, _ in inspect.getmembers(cls, callable) if not n.startswith('__')]
                print(f'  {modname}.{cname}: {methods[:20]}')
    except Exception as e:
        print(f'  {modname}: skip ({e})')

print('\n=== __init__.py ===')
init = os.path.join(pkg_dir, '__init__.py')
print(open(init).read() if os.path.exists(init) else 'no __init__.py')

In [ ]:
# ── CELL 5: Clone/update brain-neuro backend ──────────────────────────────
import os
if os.path.exists('/content/brain-neuro'):
    !git -C /content/brain-neuro pull -q
    print('brain-neuro updated')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git /content/brain-neuro
    print('brain-neuro cloned')

os.chdir('/content/brain-neuro')
print(f'cwd: {os.getcwd()}')

In [ ]:
# ── CELL 6: Download model weights ────────────────────────────────────────
from google.colab import userdata
import huggingface_hub

try:
    hf_token = userdata.get('HF_TOKEN')
    print('Using HF_TOKEN from Colab secrets')
except Exception:
    hf_token = input('Enter your HuggingFace token: ')

huggingface_hub.login(token=hf_token, add_to_git_credential=False)
huggingface_hub.hf_hub_download(
    repo_id='facebook/tribev2',
    filename='best.ckpt',
    local_dir='model_weights'
)
print('Weights downloaded.')

In [ ]:
# ── CELL 7: Patch config device ───────────────────────────────────────────
import re, torch

with open('model_weights/config.yaml') as f:
    cfg = f.read()

target = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg = re.sub(r'device: (cpu|cuda)', f'device: {target}', cfg)

with open('model_weights/config.yaml', 'w') as f:
    f.write(cfg)

print(f'Config patched → device: {target}')

In [ ]:
# ── CELL 8 (Optional): Upload cookies.txt for Instagram/TikTok ────────────
# Skip if only using YouTube.
from google.colab import files
import shutil
print('Upload cookies.txt or skip this cell')
uploaded = files.upload()
for fname in uploaded:
    shutil.move(fname, 'cookies.txt')
    print('cookies.txt saved')

In [ ]:
# ── CELL 9: Install cloudflared ───────────────────────────────────────────
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ready')

In [ ]:
# ── CELL 10: Start backend + tunnel ───────────────────────────────────────
import nest_asyncio, uvicorn, threading, subprocess, time, re as _re, os

os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

t = threading.Thread(target=run_server, daemon=True)
t.start()
time.sleep(4)
print('FastAPI started on port 8000')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = _re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n=========================================')
        print(f'  BACKEND URL: {url}')
        print(f'=========================================')
        print('Paste this into the frontend Backend field.')
        break